# Iris — Classification Benchmark (Dataset #2)

**Domain:** Biology | **Task:** Classify iris species from petal/sepal measurements

Workflow: **baseline → primary → one advanced → single comparison table → export → resume bullet.**

- **Baseline:** most-frequent predictor (DummyClassifier)
- **Primary:** KNN (k=5, scaled)
- **Advanced:** Logistic Regression, Decision Tree, tuned KNN (GridSearchCV over k)
- **Metrics:** Accuracy, macro-F1 (5-fold CV on train, hold-out test)

> Data source: UCI ML Repository `iris` (identical to Kaggle `uciml/iris`).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
# Load Iris (UCI canonical source)
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
df = pd.read_csv(URL, header=None, names=COLS)

import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/iris.csv", index=False)  # save raw copy for the Streamlit app

print(f"Rows: {len(df)}, Cols: {df.shape[1]}")
df.head()

Rows: 150, Cols: 5


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


## 1. EDA & Preprocessing

In [3]:
# Missing values (Iris is clean)
print("Missing per column:")
print(df.isna().sum())

Missing per column:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64


In [4]:
# Class balance + numeric summary
print("Class balance:")
print(df["species"].value_counts())
print()
df.describe().round(2)

Class balance:
species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64



,sepal_length,sepal_width,petal_length,petal_width
count,150.00,150.00,150.00,150.00
mean,5.84,3.05,3.76,1.20
std,0.83,0.43,1.76,0.76
min,4.30,2.00,1.00,0.10
25%,5.10,2.80,1.60,0.30
50%,5.80,3.00,4.35,1.30
75%,6.40,3.30,5.10,1.80
max,7.90,4.40,6.90,2.50


## 2. Train / Test Split + Baseline

In [5]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[feature_cols]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 120  |  Test: 30


In [6]:
# Baseline: always predict the most frequent class
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
y_base = baseline.predict(X_test)
print(f"Baseline  Accuracy={accuracy_score(y_test, y_base):.3f}  "
      f"MacroF1={f1_score(y_test, y_base, average='macro'):.3f}")

Baseline  Accuracy=0.333  MacroF1=0.167


## 3. Primary — KNN (k=5, scaled)

In [7]:
def make_model(name, est, scale=True):
    steps = []
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("clf", est))
    return name, Pipeline(steps)

knn = make_model("KNN (k=5)", KNeighborsClassifier(n_neighbors=5))

cv_rows = []
def evaluate(name, obj, use_best_cv=False):
    obj.fit(X_train, y_train)  # fit first (GridSearchCV needs this before best_score_)
    if use_best_cv:
        cv_acc = obj.best_score_
    else:
        cv_acc = cross_val_score(obj, X_train, y_train, cv=5, scoring="accuracy").mean()
    y_pred = obj.predict(X_test)
    row = {
        "model": name,
        "cv_accuracy": cv_acc,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred, average="macro"),
    }
    cv_rows.append(row)
    print(f"{name:24s} CV-Acc={cv_acc:.3f}  test Acc={row['test_accuracy']:.3f}  F1={row['test_f1']:.3f}")

name, pipe = knn
evaluate(name, pipe)

KNN (k=5)                CV-Acc=0.967  test Acc=0.933  F1=0.933

## 4. Advanced — LogReg, Decision Tree, Tuned KNN

In [8]:
advanced = [
    make_model("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("Decision Tree", DecisionTreeClassifier(random_state=RANDOM_STATE)),
]
for name, obj in advanced:
    evaluate(name, obj)

Logistic Regression      CV-Acc=0.958  test Acc=0.933  F1=0.933
Decision Tree            CV-Acc=0.942  test Acc=0.933  F1=0.933


In [9]:
# Tuned KNN: GridSearchCV over k (nested 5-fold on the train split)
tuned = Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())])
grid = GridSearchCV(tuned, param_grid={"clf__n_neighbors": list(range(1, 31))},
                    cv=5, scoring="accuracy")
evaluate("KNN (tuned k)", grid, use_best_cv=True)
print("Best k:", grid.best_params_["clf__n_neighbors"])

KNN (tuned k)            CV-Acc=0.967  test Acc=0.933  F1=0.933
Best k: 5


## 5. Comparison Table (all methods)

In [10]:
results = pd.DataFrame(cv_rows).sort_values(
    ["test_accuracy", "test_f1", "cv_accuracy"], ascending=False).reset_index(drop=True)
results.insert(0, "rank", results.index + 1)
results.round(3)

,rank,model,cv_accuracy,test_accuracy,test_f1
0,1,Logistic Regression,0.958,0.933,0.933
1,2,Decision Tree,0.942,0.933,0.933
2,3,KNN (k=5),0.967,0.933,0.933
3,4,KNN (tuned k),0.967,0.933,0.933


In [11]:
best = results.iloc[0]
print("BEST MODEL:", best["model"])
print(f"  CV-Acc = {best['cv_accuracy']:.3f} | Test Acc = {best['test_accuracy']:.3f} | "
      f"Test Macro-F1 = {best['test_f1']:.3f}")

BEST MODEL: Logistic Regression
  CV-Acc = 0.958 | Test Acc = 0.933 | Test Macro-F1 = 0.933


## 6. Export Best Model (for Streamlit deployment)

In [12]:
import joblib, json

# Fitted objects from earlier cells (Pipeline and GridSearchCV both support .fit)
fitted_lookup = {}
for name, obj in [knn, *advanced]:
    fitted_lookup[name] = obj
fitted_lookup["KNN (tuned k)"] = grid

best_obj = fitted_lookup[best["model"]]
best_obj.fit(X, y)  # refit on full dataset (metrics above stay hold-out based)

os.makedirs("models", exist_ok=True)
joblib.dump(best_obj, "models/best_model.joblib")

# Save comparison table for the Streamlit dashboard
results.to_csv("models/comparison.csv", index=False)

meta = {
    "dataset": "Iris",
    "target": "species",
    "classes": sorted(y.unique().tolist()),
    "features": feature_cols,
    "best_model": best["model"],
    "metrics": {
        "cv_accuracy": round(float(best["cv_accuracy"]), 3),
        "test_accuracy": round(float(best["test_accuracy"]), 3),
        "test_f1": round(float(best["test_f1"]), 3),
    },
    "n_samples": int(len(df)),
    "loader": "joblib.load('models/best_model.joblib')",
}
with open("models/model_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved models/best_model.joblib + models/comparison.csv + models/model_metadata.json")
print(json.dumps(meta, indent=2))

Saved models/best_model.joblib + models/comparison.csv + models/model_metadata.json
{
  "dataset": "Iris",
  "target": "species",
  "classes": [
    "Iris-setosa",
    "Iris-versicolor",
    "Iris-virginica"
  ],
  "features": [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
  ],
  "best_model": "Logistic Regression",
  "metrics": {
    "cv_accuracy": 0.958,
    "test_accuracy": 0.933,
    "test_f1": 0.933
  },
  "n_samples": 150,
  "loader": "joblib.load('models/best_model.joblib')"
}


## 7. Resume Bullet

In [13]:
bullet = (
    f"Benchmarked {len(results)} classification methods on Iris ({len(df)} samples, "
    f"{len(feature_cols)} features, 3 classes); achieved {best['model']} at test accuracy "
    f"{best['test_accuracy']:.3f} (macro-F1 {best['test_f1']:.3f})."
)
print("Resume bullet (Iris):")
print("\u2022 " + bullet)

Resume bullet (Iris):
• Benchmarked 4 classification methods on Iris (150 samples, 4 features, 3 classes); achieved Logistic Regression at test accuracy 0.933 (macro-F1 0.933).
